# 02 - Load, split and clean (leakage-safe)

This notebook runs the reusable pipeline in `src/secom` on the real data:

1. `load_secom`: join sensors and labels, sort by time
2. `split_by_time`: first 70% of units for training, last 30% for testing
3. `fit_cleaning`: learn the cleaning rules from the **training period only**
4. `apply_cleaning`: apply the same fixed rules to train and test

It then compares the result with the baseline in notebook 01, whose rules used all rows.

In [1]:
from secom.data import load_secom, split_by_time
from secom.cleaning import fit_cleaning, apply_cleaning
import pandas as pd

df = load_secom("../data/raw")
train, test = split_by_time(df)
rules = fit_cleaning(train)
train_clean = apply_cleaning(train, rules)
test_clean = apply_cleaning(test, rules)

## Check the output

Both tables should have the same columns and no missing sensor values.

In [2]:
print("train:", train_clean.shape, "test:", test_clean.shape)
print("same columns:", list(train_clean.columns) == list(test_clean.columns))
print("missing values left:", train_clean.isna().sum().sum(), test_clean.isna().sum().sum())

train: (1096, 418) test: (471, 418)
same columns: True
missing values left: 0 0


## How many sensors were dropped, and why

`fit_cleaning` only returns what it keeps, so the two drop counts are recomputed here
with the same rules: over 20% missing, then at most one distinct value.

In [3]:
sensor_cols = [c for c in train.columns if c.startswith("sensor_")]
missing_pct = train[sensor_cols].isna().mean()
n_high_missing = (missing_pct > 0.2).sum()

survivors = missing_pct[missing_pct <= 0.2].index
n_distinct = train[survivors].nunique()
n_constant = (n_distinct <= 1).sum()

n_kept = len(rules["keep_cols"])
print("dropped for missing:", n_high_missing)
print("dropped as constant:", n_constant)
print("kept:", n_kept)
print("total:", n_high_missing + n_constant + n_kept)  # must be 590

dropped for missing: 52
dropped as constant: 122
kept: 416
total: 590


## Comparison with the baseline

| | Baseline (all rows) | Leakage-safe (train only) | Difference |
|---|---|---|---|
| Dropped for missing | 32 | 52 | +20 |
| Dropped as constant | 116 | 122 | +6 |
| Kept | 442 | 416 | -26 |

**Why the numbers moved (my own words):** when we split the rows, the rules are
calculated on the training period only. The share of missing values, and which
sensors are constant, depend on how the data is distributed over time, so the
results change. They could have gone up, down or stayed the same; here both
drop counts went up.

## Which sensors crossed the 20% line?

These are the sensors that are over 20% missing in train but not over 20% overall.
The baseline kept them; the leakage-safe rules drop them.

In [4]:
baseline_missing = df[sensor_cols].isna().mean()
moved = missing_pct[(missing_pct > 0.2) & (baseline_missing <= 0.2)].index
print(len(moved))

test_missing = test[moved].isna().mean()
pd.DataFrame({
    "train": missing_pct[moved],
    "test": test_missing,
    "all rows": baseline_missing[moved],
}).round(2)

20


,train,test,all rows
sensor_546,0.23,0.01,0.17
sensor_547,0.23,0.01,0.17
sensor_548,0.23,0.01,0.17
sensor_549,0.23,0.01,0.17
sensor_550,0.23,0.01,0.17
sensor_551,0.23,0.01,0.17
sensor_552,0.23,0.01,0.17
sensor_553,0.23,0.01,0.17
sensor_554,0.23,0.01,0.17
sensor_555,0.23,0.01,0.17


**What this shows:** all 20 sensors are about 23-25% missing in train and almost
never missing in test (0-1%). Their gaps are concentrated in the early period, so
averaging over all rows (17%) hid the problem. The baseline kept them only because
it could see the later, mostly complete data: a decision based on the future.

They come in two consecutive blocks (546-557 and 562-569), which suggests one
group of measurements that was not recorded at the start, not 20 separate faults.
This is a hypothesis to keep in mind for root-cause work, not a confirmed fact.